In [1]:
import torch
import torch.nn.functional as F
import triton
import triton.language as tl

# ==========================================
# 1. The Optimized Triton Kernel (AWA)
# ==========================================

@triton.jit
def awa_padded_kernel(
    Q, K, V, Meta, Out,
    stride_qb, stride_qh, stride_ql, stride_qd,
    stride_kb, stride_kh, stride_kl, stride_kd,
    stride_vb, stride_vh, stride_vl, stride_vd,
    stride_mb, stride_mh, stride_mm, stride_md,
    stride_ob, stride_oh, stride_ol, stride_od,
    B, H, L, D, W, 
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_D: tl.constexpr,
    BLOCK_M_META: tl.constexpr, 
    M: tl.constexpr,  
):
    off_m_block = tl.program_id(0)
    off_bh = tl.program_id(1)
    off_h = off_bh % H
    off_b = off_bh // H

    # Pointers Setup
    offs_m = off_m_block * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, BLOCK_D)
    mask_m = offs_m < L
    
    Q_base = Q + (off_b * stride_qb + off_h * stride_qh)
    K_base = K + (off_b * stride_kb + off_h * stride_kh)
    V_base = V + (off_b * stride_vb + off_h * stride_vh)
    Meta_base = Meta + (off_b * stride_mb + off_h * stride_mh)
    Out_base = Out + (off_b * stride_ob + off_h * stride_oh)

    # Load Query
    q_ptr = Q_base + (offs_m[:, None] * stride_ql + offs_d[None, :] * stride_qd)
    q = tl.load(q_ptr, mask=mask_m[:, None] & (offs_d[None, :] < D), other=0.0)
    q = q * (1.0 / tl.sqrt(D.to(tl.float32)))

    # --- Meta Sink ---
    offs_meta_m = tl.arange(0, BLOCK_M_META) 
    meta_ptr = Meta_base + (offs_meta_m[None, :] * stride_mm + offs_d[:, None] * stride_md)
    meta_t = tl.load(meta_ptr, mask=(offs_meta_m[None, :] < M) & (offs_d[:, None] < D), other=0.0)
    
    meta_scores = tl.dot(q.to(tl.float16), meta_t.to(tl.float16))
    meta_scores = tl.where(offs_meta_m[None, :] < M, meta_scores, float("-inf"))
    
    m_i = tl.max(meta_scores, 1)        
    l_i = tl.sum(tl.exp(meta_scores - m_i[:, None]), 1) 
    acc = tl.zeros([BLOCK_M, BLOCK_D], dtype=tl.float32)

    # --- Sliding Window ---
    start_m_idx = off_m_block * BLOCK_M
    min_k_idx = tl.maximum(0, start_m_idx - W + 1)
    max_k_idx = (off_m_block + 1) * BLOCK_M
    
    start_block_n = min_k_idx // BLOCK_N
    end_block_n = tl.cdiv(max_k_idx, BLOCK_N)
    offs_n_base = tl.arange(0, BLOCK_N)
    
    for block_n in range(start_block_n, end_block_n):
        start_n = block_n * BLOCK_N
        offs_n = start_n + offs_n_base
        k_ptr = K_base + (offs_n[None, :] * stride_kl + offs_d[:, None] * stride_kd)
        k_t = tl.load(k_ptr, mask=(offs_n[None, :] < L) & (offs_d[:, None] < D), other=0.0)
        
        qk = tl.dot(q.to(tl.float16), k_t.to(tl.float16))
        
        dist = offs_m[:, None] - offs_n[None, :]
        mask_val = (dist >= 0) & (dist < W)
        qk = tl.where(mask_val, qk, float("-inf"))
        
        m_curr = tl.max(qk, 1)                 
        m_new = tl.maximum(m_i, m_curr)        
        p = tl.exp(qk - m_new[:, None])
        alpha = tl.exp(m_i - m_new)
        l_i = l_i * alpha + tl.sum(p, 1)
        
        v_ptr = V_base + (offs_n[:, None] * stride_vl + offs_d[None, :] * stride_vd)
        v = tl.load(v_ptr, mask=(offs_n[:, None] < L) & (offs_d[None, :] < D), other=0.0)
        
        acc = acc * alpha[:, None]
        acc += tl.dot(p.to(tl.float16), v.to(tl.float16))
        m_i = m_new

    # --- Store ---
    out = acc / (l_i[:, None] + 1e-8)
    out_ptr = Out_base + (offs_m[:, None] * stride_ol + offs_d[None, :] * stride_od)
    tl.store(out_ptr, out.to(tl.float16), mask=mask_m[:, None] & (offs_d[None, :] < D))

def run_awa_triton(q, k, v, meta_tokens, window_size):
    # Enforce (B, H, L, D)
    if q.shape[1] > q.shape[2]: # Heuristic: if dim 1 is larger than dim 2, assume (B, L, H, D)
         q = q.transpose(1, 2).contiguous()
         k = k.transpose(1, 2).contiguous()
         v = v.transpose(1, 2).contiguous()
         transposed = True
    else:
         q, k, v = q.contiguous(), k.contiguous(), v.contiguous()
         transposed = False
         
    B, H, L, D = q.shape
    M_val = meta_tokens.shape[0]
    out = torch.empty_like(q)
    stride_m, stride_d = meta_tokens.stride()

    BLOCK_M = 64
    BLOCK_N = 64
    BLOCK_D = triton.next_power_of_2(D)
    next_pow2_M = triton.next_power_of_2(M_val)
    BLOCK_M_META = max(16, next_pow2_M)

    grid = (triton.cdiv(L, BLOCK_M), B * H)
    
    awa_padded_kernel[grid](
        q, k, v, meta_tokens, out,
        *q.stride(), *k.stride(), *v.stride(),
        0, 0, stride_m, stride_d,
        *out.stride(),
        B, H, L, D, window_size,
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_D=BLOCK_D,
        BLOCK_M_META=BLOCK_M_META, M=M_val,
        num_stages=3, num_warps=4 
    )
    
    return out.transpose(1, 2) if transposed else out


# ==========================================
# 2. PyTorch Reference Implementation
# ==========================================

def pytorch_awa_reference(q, k, v, meta_tokens, window_size):
    """
    Ref implementation of Anchor Window Attention.
    Inputs:
      q, k, v: (B, L, H, D)
      meta_tokens: (M, D)
    """
    # 1. Setup
    B, L, H, D = q.shape
    scale = 1.0 / (D ** 0.5)
    
    # Move to (B, H, L, D) for easier processing
    q = q.transpose(1, 2) 
    k = k.transpose(1, 2)
    v = v.transpose(1, 2)
    
    # 2. Global Anchor Scores (The Sink)
    # meta_tokens: (M, D) -> broadcast to (B, H, M, D)
    # q: (B, H, L, D)
    # scores: (B, H, L, M)
    meta_scores = torch.matmul(q, meta_tokens.t()) * scale
    
    # 3. Local Sliding Window Scores
    # Pad Left so window at 't' sees [t-w+1 ... t]
    pad_left = window_size - 1
    k_padded = F.pad(k, (0, 0, pad_left, 0)) # Pad seq dim
    v_padded = F.pad(v, (0, 0, pad_left, 0))
    
    # Unfold: (B, H, L, D, W)
    k_win = k_padded.unfold(2, window_size, 1)
    v_win = v_padded.unfold(2, window_size, 1)
    
    # Local Scores: (B, H, L, 1, D) @ (B, H, L, D, W) -> (B, H, L, 1, W)
    local_scores = torch.matmul(q.unsqueeze(-2), k_win).squeeze(-2) * scale
    
    # Masking padding (where index < 0 relative to original seq)
    # We use a simple mask: valid if index >= 0
    # The unfold naturally aligns, but we need to mask the "pre-sequence" padding
    # Actually, F.pad with 0s puts 0 vectors. 
    # But strictly, we should mask positions where loop index < window_size-1.
    # However, for correctness check, usually masking with -inf is preferred.
    mask = torch.ones(L, device=q.device)
    mask_padded = F.pad(mask, (pad_left, 0), value=0)
    mask_win = mask_padded.unfold(0, window_size, 1) # (L, W)
    local_scores = local_scores.masked_fill(mask_win.unsqueeze(0).unsqueeze(0) == 0, float("-inf"))

    # 4. Normalization (Sink Logic)
    # Combine to compute max for stability
    # shapes: meta (..., M), local (..., W)
    max_meta = meta_scores.max(dim=-1, keepdim=True)[0]
    max_local = local_scores.max(dim=-1, keepdim=True)[0]
    max_val = torch.maximum(max_meta, max_local)
    
    exp_meta = torch.exp(meta_scores - max_val) # (B, H, L, M)
    exp_local = torch.exp(local_scores - max_val) # (B, H, L, W)
    
    # Denominator includes BOTH
    denom = exp_meta.sum(dim=-1, keepdim=True) + exp_local.sum(dim=-1, keepdim=True)
    
    # 5. Output (Numerator only has Local)
    # exp_local: (..., W) -> (..., 1, W)
    # v_win: (..., D, W) -> Transpose to (..., W, D)
    # Result: (..., 1, D)
    numerator = torch.matmul(exp_local.unsqueeze(-2), v_win.transpose(-1, -2)).squeeze(-2)
    
    out = numerator / (denom + 1e-8)
    return out.transpose(1, 2) # Return to (B, L, H, D)


# ==========================================
# 3. Main Benchmark Script
# ==========================================

B, L, H, D = 2, 1024, 4, 128
WINDOW_SIZE = 128
NUM_META = 4 # Small number (problematic case previously)
dtype = torch.float16 # Use FP16 for Triton, FP32 for Reference to verify precision
device = "cuda"

print(f"Benchmarking AWA: Batch={B}, Len={L}, Heads={H}, Dim={D}")
print(f"Config: Window={WINDOW_SIZE}, MetaTokens={NUM_META}")

torch.manual_seed(0)
# Create FP32 inputs for reference
q_fp32 = torch.randn((B, L, H, D), device=device, dtype=torch.float32)
k_fp32 = torch.randn((B, L, H, D), device=device, dtype=torch.float32)
v_fp32 = torch.randn((B, L, H, D), device=device, dtype=torch.float32)
meta_fp32 = torch.randn((NUM_META, D), device=device, dtype=torch.float32)

# Create FP16 inputs for Triton
q_fp16 = q_fp32.half()
k_fp16 = k_fp32.half()
v_fp16 = v_fp32.half()
meta_fp16 = meta_fp32.half()

print("\nVerifying Correctness...")
print("Running PyTorch Reference (FP32)...")
ref_out = pytorch_awa_reference(q_fp32, k_fp32, v_fp32, meta_fp32, WINDOW_SIZE)

print("Running Triton Kernel (FP16)...")
tri_out = run_awa_triton(q_fp16, k_fp16, v_fp16, meta_fp16, WINDOW_SIZE)

# Compare
diff = (ref_out - tri_out.float()).abs()
max_diff = diff.max().item()
mean_val = ref_out.abs().mean().item()

print(f"   Output Mean Value: {mean_val:.4f}")
print(f"   Max Difference:    {max_diff:.4f}")
print(f"   Rel Difference:    {100 * max_diff / mean_val:.2f}%")

if max_diff < 0.1: # Relaxed tolerance for FP16 vs FP32 attention
    print("PASS: Correctness verified (within FP16 tolerance).")
else:
    print("FAIL: Difference is too high.")

print("\nStarting Performance Benchmark (Triton)...")

# Warmup
for _ in range(10):
    _ = run_awa_triton(q_fp16, k_fp16, v_fp16, meta_fp16, WINDOW_SIZE)
torch.cuda.synchronize()

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

start_event.record()
n_loops = 100
for _ in range(n_loops):
    _ = run_awa_triton(q_fp16, k_fp16, v_fp16, meta_fp16, WINDOW_SIZE)
end_event.record()
torch.cuda.synchronize()

elapsed_time_ms = start_event.elapsed_time(end_event)
print(f"Average time per run: {elapsed_time_ms / n_loops:.4f} ms")

Benchmarking AWA: Batch=2, Len=1024, Heads=4, Dim=128
Config: Window=128, MetaTokens=4

Verifying Correctness...
Running PyTorch Reference (FP32)...
Running Triton Kernel (FP16)...
   Output Mean Value: 0.1155
   Max Difference:    0.0012
   Rel Difference:    1.01%
PASS: Correctness verified (within FP16 tolerance).

Starting Performance Benchmark (Triton)...
Average time per run: 0.1217 ms


In [3]:
import torch
import triton
import triton.language as tl
import torch.nn.functional as F

# ==========================================
# 1. The Triton Kernel (TF32 Version)
# ==========================================

@triton.jit
def awa_tf32_kernel(
    Q, K, V, Meta, Out,
    stride_qb, stride_qh, stride_ql, stride_qd,
    stride_kb, stride_kh, stride_kl, stride_kd,
    stride_vb, stride_vh, stride_vl, stride_vd,
    stride_mb, stride_mh, stride_mm, stride_md,
    stride_ob, stride_oh, stride_ol, stride_od,
    B, H, L, D, W, 
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_D: tl.constexpr,
    BLOCK_M_META: tl.constexpr, 
    M: tl.constexpr,  
):
    off_m_block = tl.program_id(0)
    off_bh = tl.program_id(1)
    off_h = off_bh % H
    off_b = off_bh // H

    # Pointers Setup
    offs_m = off_m_block * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, BLOCK_D)
    mask_m = offs_m < L
    
    Q_base = Q + (off_b * stride_qb + off_h * stride_qh)
    K_base = K + (off_b * stride_kb + off_h * stride_kh)
    V_base = V + (off_b * stride_vb + off_h * stride_vh)
    Meta_base = Meta + (off_b * stride_mb + off_h * stride_mh)
    Out_base = Out + (off_b * stride_ob + off_h * stride_oh)

    # Load Query as FP32 (Triton keeps it in registers as FP32)
    q_ptr = Q_base + (offs_m[:, None] * stride_ql + offs_d[None, :] * stride_qd)
    q = tl.load(q_ptr, mask=mask_m[:, None] & (offs_d[None, :] < D), other=0.0)
    q = q * (1.0 / tl.sqrt(D.to(tl.float32)))

    # --- Meta Sink ---
    offs_meta_m = tl.arange(0, BLOCK_M_META) 
    meta_ptr = Meta_base + (offs_meta_m[None, :] * stride_mm + offs_d[:, None] * stride_md)
    meta_t = tl.load(meta_ptr, mask=(offs_meta_m[None, :] < M) & (offs_d[:, None] < D), other=0.0)
    
    # TF32 MAGIC: tl.dot(fp32, fp32) automatically uses TF32 on L4/A100
    meta_scores = tl.dot(q, meta_t)
    meta_scores = tl.where(offs_meta_m[None, :] < M, meta_scores, float("-inf"))
    
    m_i = tl.max(meta_scores, 1)        
    l_i = tl.sum(tl.exp(meta_scores - m_i[:, None]), 1) 
    acc = tl.zeros([BLOCK_M, BLOCK_D], dtype=tl.float32)

    # --- Sliding Window ---
    start_m_idx = off_m_block * BLOCK_M
    min_k_idx = tl.maximum(0, start_m_idx - W + 1)
    max_k_idx = (off_m_block + 1) * BLOCK_M
    
    start_block_n = min_k_idx // BLOCK_N
    end_block_n = tl.cdiv(max_k_idx, BLOCK_N)
    offs_n_base = tl.arange(0, BLOCK_N)
    
    for block_n in range(start_block_n, end_block_n):
        start_n = block_n * BLOCK_N
        offs_n = start_n + offs_n_base
        k_ptr = K_base + (offs_n[None, :] * stride_kl + offs_d[:, None] * stride_kd)
        k_t = tl.load(k_ptr, mask=(offs_n[None, :] < L) & (offs_d[:, None] < D), other=0.0)
        
        # TF32 Dot
        qk = tl.dot(q, k_t)
        
        dist = offs_m[:, None] - offs_n[None, :]
        mask_val = (dist >= 0) & (dist < W)
        qk = tl.where(mask_val, qk, float("-inf"))
        
        m_curr = tl.max(qk, 1)                 
        m_new = tl.maximum(m_i, m_curr)        
        p = tl.exp(qk - m_new[:, None])
        alpha = tl.exp(m_i - m_new)
        l_i = l_i * alpha + tl.sum(p, 1)
        
        v_ptr = V_base + (offs_n[:, None] * stride_vl + offs_d[None, :] * stride_vd)
        v = tl.load(v_ptr, mask=(offs_n[:, None] < L) & (offs_d[None, :] < D), other=0.0)
        
        acc = acc * alpha[:, None]
        # TF32 Dot (Accumulate in FP32)
        acc += tl.dot(p.to(tl.float32), v)
        m_i = m_new

    # --- Store ---
    out = acc / (l_i[:, None] + 1e-8)
    out_ptr = Out_base + (offs_m[:, None] * stride_ol + offs_d[None, :] * stride_od)
    tl.store(out_ptr, out, mask=mask_m[:, None] & (offs_d[None, :] < D))

# ==========================================
# 2. Wrapper
# ==========================================

def run_awa_tf32(q, k, v, meta_tokens, window_size):
    # Ensure standard (B, H, L, D) layout
    if q.shape[1] > q.shape[2]: # Heuristic for (B, L, H, D)
         q = q.transpose(1, 2).contiguous()
         k = k.transpose(1, 2).contiguous()
         v = v.transpose(1, 2).contiguous()
         transposed = True
    else:
         q, k, v = q.contiguous(), k.contiguous(), v.contiguous()
         transposed = False

    # Force inputs to Float32 for TF32 usage
    if q.dtype != torch.float32:
        q = q.float()
        k = k.float()
        v = v.float()
        meta_tokens = meta_tokens.float()

    B, H, L, D = q.shape
    M_val = meta_tokens.shape[0]
    out = torch.empty_like(q)
    stride_m, stride_d = meta_tokens.stride()

    # L4 Tuning: 64 or 128 usually works well. 
    # For TF32, registers pressure is higher, so 64 is safer.
    BLOCK_M = 32
    BLOCK_N = 32
    BLOCK_D = triton.next_power_of_2(D)
    next_pow2_M = triton.next_power_of_2(M_val)
    BLOCK_M_META = max(16, next_pow2_M)

    grid = (triton.cdiv(L, BLOCK_M), B * H)
    
    awa_tf32_kernel[grid](
        q, k, v, meta_tokens, out,
        *q.stride(), *k.stride(), *v.stride(),
        0, 0, stride_m, stride_d,
        *out.stride(),
        B, H, L, D, window_size,
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_D=BLOCK_D,
        BLOCK_M_META=BLOCK_M_META, M=M_val,
        num_stages=3, num_warps=4 
    )
    
    return out.transpose(1, 2) if transposed else out

# ==========================================
# 3. Reference (Strict FP32 Pytorch)
# ==========================================

def pytorch_baseline(q, k, v, meta_tokens, window_size):
    # Standard PyTorch implementation for verification
    # q, k, v input shape (B, L, H, D) expected here
    B, L, H, D = q.shape
    scale = 1.0 / (D ** 0.5)
    
    q = q.transpose(1, 2) # (B, H, L, D)
    k = k.transpose(1, 2)
    v = v.transpose(1, 2)
    
    # Meta Scores
    meta_scores = torch.matmul(q, meta_tokens.t()) * scale
    
    # Window Scores
    pad_left = window_size - 1
    k_padded = F.pad(k, (0, 0, pad_left, 0))
    v_padded = F.pad(v, (0, 0, pad_left, 0))
    k_win = k_padded.unfold(2, window_size, 1)
    v_win = v_padded.unfold(2, window_size, 1)
    
    local_scores = torch.matmul(q.unsqueeze(-2), k_win).squeeze(-2) * scale
    
    mask = torch.ones(L, device=q.device)
    mask_padded = F.pad(mask, (pad_left, 0), value=0)
    mask_win = mask_padded.unfold(0, window_size, 1)
    local_scores = local_scores.masked_fill(mask_win.unsqueeze(0).unsqueeze(0) == 0, float("-inf"))

    max_meta = meta_scores.max(dim=-1, keepdim=True)[0]
    max_local = local_scores.max(dim=-1, keepdim=True)[0]
    max_val = torch.maximum(max_meta, max_local)
    
    exp_meta = torch.exp(meta_scores - max_val)
    exp_local = torch.exp(local_scores - max_val)
    
    denom = exp_meta.sum(dim=-1, keepdim=True) + exp_local.sum(dim=-1, keepdim=True)
    numerator = torch.matmul(exp_local.unsqueeze(-2), v_win.transpose(-1, -2)).squeeze(-2)
    
    out = numerator / (denom + 1e-8)
    return out.transpose(1, 2)

# ==========================================
# 4. Benchmark Script
# ==========================================

if __name__ == "__main__":
    B, L, H, D = 2, 1024, 4, 128
    WINDOW_SIZE = 128
    NUM_META = 4
    dtype = torch.float32 
    device = "cuda"

    print(f"Benchmarking AWA (TF32 on L4): Batch={B}, Len={L}, Heads={H}, Dim={D}")
    
    torch.manual_seed(0)
    q = torch.randn((B, L, H, D), device=device, dtype=dtype)
    k = torch.randn((B, L, H, D), device=device, dtype=dtype)
    v = torch.randn((B, L, H, D), device=device, dtype=dtype)
    meta = torch.randn((NUM_META, D), device=device, dtype=dtype)

    print("\nVerifying Correctness (Ref FP32 vs Triton TF32)...")
    
    # PyTorch Baseline
    ref_out = pytorch_baseline(q, k, v, meta, WINDOW_SIZE)
    
    # Triton TF32
    # Ensure allow_tf32 is strictly enabled for PyTorch comparisons if mixing ops,
    # but Triton handles it internally via the kernel.
    torch.backends.cuda.matmul.allow_tf32 = True 
    tri_out = run_awa_tf32(q, k, v, meta, WINDOW_SIZE)

    diff = (ref_out - tri_out).abs()
    max_diff = diff.max().item()
    mean_val = ref_out.abs().mean().item()

    print(f"   Output Mean Value: {mean_val:.6f}")
    print(f"   Max Difference:    {max_diff:.6f}")
    
    # TF32 precision is better than FP16 but worse than strict FP32
    # Expect diff around 1e-4 or 1e-5
    if max_diff < 5e-4: 
        print("PASS: High Precision verified.")
    else:
        print(f"Diff is {100 * max_diff / mean_val:.3f}% of mean value.")

    print("\nStarting Performance Benchmark...")

    # Warmup
    for _ in range(10):
        _ = run_awa_tf32(q, k, v, meta, WINDOW_SIZE)
    torch.cuda.synchronize()

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    start_event.record()
    n_loops = 100
    for _ in range(n_loops):
        _ = run_awa_tf32(q, k, v, meta, WINDOW_SIZE)
    end_event.record()
    torch.cuda.synchronize()

    elapsed_time_ms = start_event.elapsed_time(end_event)
    avg_time = elapsed_time_ms / n_loops
    print(f"Average time per run: {avg_time:.4f} ms")

Benchmarking AWA (TF32 on L4): Batch=2, Len=1024, Heads=4, Dim=128

Verifying Correctness (Ref FP32 vs Triton TF32)...
   Output Mean Value: 0.115491
   Max Difference:    0.003074
Diff is 2.662% of mean value.

Starting Performance Benchmark...
Average time per run: 0.1201 ms
